In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import  mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor

In [2]:
df = pd.read_csv("../data/processed/train_cleaned.csv")

In [3]:
X = df.drop('SalePrice', axis=1)
y = df['SalePrice']

In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [5]:
numeric_cols = X_train.select_dtypes(include=['int64', 'float64']).columns
categorical_cols = X_train.select_dtypes(include=['object']).columns

In [6]:
scaler = StandardScaler()

In [7]:
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

In [8]:
X_train_num = scaler.fit_transform(X_train[numeric_cols])
X_test_num = scaler.transform(X_test[numeric_cols])

In [9]:
X_train_cat = encoder.fit_transform(X_train[categorical_cols])
X_test_cat = encoder.transform(X_test[categorical_cols])

In [10]:
cat_feature_names = encoder.get_feature_names_out(categorical_cols)

X_train_final = pd.DataFrame(np.hstack([X_train_num, X_train_cat]), columns=list(numeric_cols) + list(cat_feature_names))
X_test_final = pd.DataFrame(np.hstack([X_test_num, X_test_cat]), columns=list(numeric_cols) + list(cat_feature_names))

In [11]:
lr = LinearRegression()
lr.fit(X_train_final, y_train)

y_lr = lr.predict(X_test_final)

In [12]:
ridge = Ridge(alpha=1.0)
ridge.fit(X_train_final, y_train)
y_ridge = ridge.predict(X_test_final)

lasso = Lasso(alpha=0.001)
lasso.fit(X_train_final, y_train)
y_lasso = lasso.predict(X_test_final)

In [13]:
print("Training Random Forest...")
rf = RandomForestRegressor(n_estimators=500, random_state=42)
rf.fit(X_train_final, y_train)

Training Random Forest...


RandomForestRegressor(n_estimators=500, random_state=42)

In [14]:
y_pred_log = rf.predict(X_test_final)
y_pred = np.expm1(y_pred_log)
y_test_actual = np.expm1(y_test)

In [15]:
rmse = np.sqrt(mean_squared_error(y_test_actual, y_pred))
r2 = r2_score(y_test_actual, y_pred)

print(f"Random Forest -> RMSE: {rmse:.4f}, R2: {r2:.4f}")

Random Forest -> RMSE: 24002.8721, R2: 0.8957


In [16]:
def evaluate(y_true, y_pred, name="Model"):
    rmse = np.sqrt(mean_squared_error(y_test_actual, y_pred))
    r2 = r2_score(y_test_actual, y_pred)
    print(f"{name} -> RMSE: {rmse:.4f}, R2: {r2:.4f}")

evaluate(y_test, y_lr, "Linear Regression")
evaluate(y_test, y_ridge, "Ridge Regression")
evaluate(y_test, y_lasso, "Lasso Regression")
evaluate(y_test, y_pred_log, "Random Forest")

evaluate(y_test_actual, y_pred, "Random Forest")

Linear Regression -> RMSE: 195995.3768, R2: -5.9544
Ridge Regression -> RMSE: 195995.3733, R2: -5.9544
Lasso Regression -> RMSE: 195995.3764, R2: -5.9544
Random Forest -> RMSE: 195995.3827, R2: -5.9544
Random Forest -> RMSE: 24002.8721, R2: 0.8957


In [17]:
import xgboost as xgb

In [18]:
print("Training XGBoost...")
xgb_model = xgb.XGBRegressor(
    objective='reg:squarederror',
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=5,
    n_jobs=-1,
    random_state=42
)

Training XGBoost...


In [19]:
xgb_model.fit(X_train_final, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.05, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=5,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=1000,
             n_jobs=-1, num_parallel_tree=None, ...)

In [20]:
y_pred_log = xgb_model.predict(X_test_final)
y_pred = np.expm1(y_pred_log)
y_test_actual = np.expm1(y_test)

In [21]:
rmse = np.sqrt(mean_squared_error(y_test_actual, y_pred))
r2 = r2_score(y_test_actual, y_pred)

print(f"XGBoost Model -> RMSE: {rmse:.4f}, R2: {r2:.4f}")

XGBoost Model -> RMSE: 21687.4561, R2: 0.9148
